In [1]:
import os
import random
import cv2
import mediapipe as mp
from tqdm import tqdm

# Fix seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)

set_seed(42)

In [6]:
# Change this to your actual dataset path
real_videos_path = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23\original"
fake_videos_path = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23\Deepfakes"

In [7]:
print("Real videos:", len(os.listdir(real_videos_path)))
print("Fake videos:", len(os.listdir(fake_videos_path)))

Real videos: 1000
Fake videos: 1000


In [8]:
def split_videos(video_folder, train_ratio=0.8, max_videos=600):
    videos = sorted(os.listdir(video_folder))[:max_videos]
    random.shuffle(videos)

    train_size = int(train_ratio * len(videos))
    train_videos = videos[:train_size]
    val_videos = videos[train_size:]

    return train_videos, val_videos


# Create splits
real_train, real_val = split_videos(real_videos_path, train_ratio=0.8, max_videos=600)
fake_train, fake_val = split_videos(fake_videos_path, train_ratio=0.8, max_videos=600)

print("Real Train:", len(real_train))
print("Real Val:", len(real_val))
print("Fake Train:", len(fake_train))
print("Fake Val:", len(fake_val))

Real Train: 480
Real Val: 120
Fake Train: 480
Fake Val: 120


In [9]:
mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(
    model_selection=1,
    min_detection_confidence=0.5
)

In [10]:
def extract_faces_from_list(video_folder, video_list, output_folder, label, frame_interval=30):
    os.makedirs(os.path.join(output_folder, label), exist_ok=True)

    for video in tqdm(video_list):
        video_path = os.path.join(video_folder, video)
        cap = cv2.VideoCapture(video_path)

        frame_id = 0
        saved_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_id % frame_interval == 0:
                h, w, _ = frame.shape
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = face_detector.process(rgb)

                if results.detections:
                    detection = results.detections[0]
                    bbox = detection.location_data.relative_bounding_box

                    x1 = int(bbox.xmin * w)
                    y1 = int(bbox.ymin * h)
                    box_w = int(bbox.width * w)
                    box_h = int(bbox.height * h)

                    x2 = x1 + box_w
                    y2 = y1 + box_h

                    # Clamp
                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(w, x2)
                    y2 = min(h, y2)

                    face_crop = frame[y1:y2, x1:x2]

                    if face_crop.size > 0:
                        face_crop = cv2.resize(face_crop, (224, 224))
                        frame_name = f"{video[:-4]}_{saved_count}.jpg"
                        save_path = os.path.join(output_folder, label, frame_name)
                        cv2.imwrite(save_path, face_crop)
                        saved_count += 1

            frame_id += 1

        cap.release()

    print(f"Finished extracting faces for {label}")

In [11]:
output_root = r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_faces"

In [12]:
# TRAIN SET
extract_faces_from_list(real_videos_path, real_train,
                        os.path.join(output_root, "train"), "real")

extract_faces_from_list(fake_videos_path, fake_train,
                        os.path.join(output_root, "train"), "fake")

# VALIDATION SET
extract_faces_from_list(real_videos_path, real_val,
                        os.path.join(output_root, "val"), "real")

extract_faces_from_list(fake_videos_path, fake_val,
                        os.path.join(output_root, "val"), "fake")

C:\Users\jnana\anaconda3\envs\mediapipe_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 480/480 [22:00<00:00,  2.75s/it]


Finished extracting faces for real


100%|██████████| 480/480 [21:25<00:00,  2.68s/it]


Finished extracting faces for fake


100%|██████████| 120/120 [05:44<00:00,  2.87s/it]


Finished extracting faces for real


100%|██████████| 120/120 [06:21<00:00,  3.18s/it]

Finished extracting faces for fake


In [14]:
def count_images(folder):
    return len([f for f in os.listdir(folder) if f.endswith(".jpg")])

print("TRAIN REAL:", count_images(r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_faces\train\real"))
print("TRAIN FAKE:", count_images(r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_faces\train\fake"))
print("VAL REAL:", count_images(r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_faces\val\real"))
print("VAL FAKE:", count_images(r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_faces\val\fake"))

TRAIN REAL: 8544
TRAIN FAKE: 8360
VAL REAL: 2122
VAL FAKE: 2295


In [1]:
import numpy
print(numpy.__version__) 

1.26.4


video extrction 2

In [2]:
real_videos_path      = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23\original"
deepfakes_path        = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23\Deepfakes"
faceswap_path         = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23\FaceSwap"
face2face_path        = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23\Face2Face"

In [3]:
print("Real:", len(os.listdir(real_videos_path)))
print("Deepfakes:", len(os.listdir(deepfakes_path)))
print("FaceSwap:", len(os.listdir(faceswap_path)))
print("Face2Face:", len(os.listdir(face2face_path)))

Real: 1000
Deepfakes: 1000
FaceSwap: 1000
Face2Face: 1000


In [4]:
def split_videos(video_folder, train_ratio=0.8, max_videos=300):
    videos = sorted(os.listdir(video_folder))[:max_videos]
    random.shuffle(videos)

    train_size = int(train_ratio * len(videos))
    return videos[:train_size], videos[train_size:]


# Real
real_train, real_val = split_videos(real_videos_path, max_videos=600)

# Deepfakes
df_train, df_val = split_videos(deepfakes_path, max_videos=600)

# FaceSwap
fs_train, fs_val = split_videos(faceswap_path, max_videos=600)

# Face2Face
f2f_train, f2f_val = split_videos(face2face_path, max_videos=600)

In [5]:
mp_face = mp.solutions.face_detection
face_detector = mp_face.FaceDetection(
    model_selection=1,
    min_detection_confidence=0.5
)

In [6]:
def extract_faces_from_list(video_folder, video_list, output_folder, label, frame_interval=30):
    os.makedirs(os.path.join(output_folder, label), exist_ok=True)

    for video in tqdm(video_list):
        video_path = os.path.join(video_folder, video)
        cap = cv2.VideoCapture(video_path)

        frame_id = 0
        saved_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_id % frame_interval == 0:
                h, w, _ = frame.shape
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = face_detector.process(rgb)

                if results.detections:
                    detection = results.detections[0]
                    bbox = detection.location_data.relative_bounding_box

                    x1 = int(bbox.xmin * w)
                    y1 = int(bbox.ymin * h)
                    bw = int(bbox.width * w)
                    bh = int(bbox.height * h)

                    x2 = x1 + bw
                    y2 = y1 + bh

                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(w, x2)
                    y2 = min(h, y2)

                    face_crop = frame[y1:y2, x1:x2]

                    if face_crop.size > 0:
                        face_crop = cv2.resize(face_crop, (224, 224))
                        filename = f"{video[:-4]}_{saved_count}.jpg"
                        save_path = os.path.join(output_folder, label, filename)
                        cv2.imwrite(save_path, face_crop)
                        saved_count += 1

            frame_id += 1

        cap.release()

    print(f"Finished extracting {label}")

In [7]:
output_root = r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_multi_faces"

In [8]:
# TRAIN REAL
extract_faces_from_list(real_videos_path, real_train,
                        os.path.join(output_root, "train"), "real")

# TRAIN FAKES (combine 3 types)
extract_faces_from_list(deepfakes_path, df_train,
                        os.path.join(output_root, "train"), "fake")

extract_faces_from_list(faceswap_path, fs_train,
                        os.path.join(output_root, "train"), "fake")

extract_faces_from_list(face2face_path, f2f_train,
                        os.path.join(output_root, "train"), "fake")

C:\Users\jnana\anaconda3\envs\mediapipe_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 480/480 [22:32<00:00,  2.82s/it]


Finished extracting real


100%|██████████| 480/480 [22:37<00:00,  2.83s/it]


Finished extracting fake


100%|██████████| 480/480 [18:44<00:00,  2.34s/it]


Finished extracting fake


100%|██████████| 480/480 [22:48<00:00,  2.85s/it]

Finished extracting fake


In [9]:
# VAL REAL
extract_faces_from_list(real_videos_path, real_val,
                        os.path.join(output_root, "val"), "real")

# VAL FAKES
extract_faces_from_list(deepfakes_path, df_val,
                        os.path.join(output_root, "val"), "fake")

extract_faces_from_list(faceswap_path, fs_val,
                        os.path.join(output_root, "val"), "fake")

extract_faces_from_list(face2face_path, f2f_val,
                        os.path.join(output_root, "val"), "fake")

100%|██████████| 120/120 [05:54<00:00,  2.95s/it]


Finished extracting real


100%|██████████| 120/120 [06:29<00:00,  3.25s/it]


Finished extracting fake


100%|██████████| 120/120 [03:59<00:00,  2.00s/it]


Finished extracting fake


100%|██████████| 120/120 [05:15<00:00,  2.63s/it]

Finished extracting fake


multiclass dataset


In [3]:
!pip install mediapipe opencv-python tqdm

  Using cached mediapipe-0.10.32-py3-none-win_amd64.whl.metadata (9.8 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sounddevice-0.5.5-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached mediapipe-0.10.32-py3-none-win_amd64.whl (10.2 MB)
Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
Using cached flatbuffers-25.12.19-py2.py3-none-any.whl (26 kB)
Using cached sounddevic

In [2]:
import os
import cv2
import random
import mediapipe as mp
from tqdm import tqdm

In [3]:
ffpp_root = r"C:\Users\jnana\Downloads\deepfake\archive (1)\FaceForensics++_C23"

dataset_paths = {
    "original": os.path.join(ffpp_root, "original"),
    "deepfakes": os.path.join(ffpp_root, "Deepfakes"),
    "faceswap": os.path.join(ffpp_root, "FaceSwap"),
    "face2face": os.path.join(ffpp_root, "Face2Face"),
    "neuraltextures": os.path.join(ffpp_root, "NeuralTextures"),
    "faceshifter": os.path.join(ffpp_root, "FaceShifter"),
    "deepfakedetection": os.path.join(ffpp_root, "DeepFakeDetection")
}

In [4]:
output_root = r"C:\Users\jnana\Downloads\deepfake\archive (1)\data\ffpp_multiclass_faces"

classes = [
    "original",
    "deepfakes",
    "faceswap",
    "face2face",
    "neuraltextures",
    "faceshifter",
    "deepfakedetection"
]

#for split in ["train","val"]:
 #   for c in classes:
   #     os.makedirs(os.path.join(output_root, split, c), exist_ok=True)

print("Dataset folders created.")

Dataset folders created.


In [5]:
from mediapipe.tasks.python.vision import FaceDetector
from mediapipe.tasks.python.vision import FaceDetectorOptions
from mediapipe.tasks.python.core.base_options import BaseOptions

In [6]:
import mediapipe as mp
from mediapipe.python.solutions import face_detection
from mediapipe.python.solutions import drawing_utils
face_detector = face_detection.FaceDetection(
    model_selection=1,
    min_detection_confidence=0.5
)

In [7]:
def split_videos(video_folder, train_ratio=0.8):

    videos = [v for v in os.listdir(video_folder) if v.endswith(".mp4")]

    random.shuffle(videos)

    split_index = int(len(videos) * train_ratio)

    train_videos = videos[:split_index]
    val_videos = videos[split_index:]

    return train_videos, val_videos

In [8]:
def extract_faces_from_videos(video_folder,
                              video_list,
                              save_folder,
                              class_name,
                              frame_interval=30):

    save_dir = os.path.join(save_folder, class_name)
    os.makedirs(save_dir, exist_ok=True)

    for video in tqdm(video_list):

        video_path = os.path.join(video_folder, video)
        cap = cv2.VideoCapture(video_path)

        frame_id = 0
        saved_count = 0

        while True:

            ret, frame = cap.read()
            if not ret:
                break

            if frame_id % frame_interval == 0:

                h, w, _ = frame.shape

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                results = face_detector.process(rgb)

                if results.detections:

                    detection = results.detections[0]

                    bbox = detection.location_data.relative_bounding_box

                    x1 = int(bbox.xmin * w)
                    y1 = int(bbox.ymin * h)

                    x2 = int((bbox.xmin + bbox.width) * w)
                    y2 = int((bbox.ymin + bbox.height) * h)

                    x1 = max(0, x1)
                    y1 = max(0, y1)
                    x2 = min(w, x2)
                    y2 = min(h, y2)

                    face = frame[y1:y2, x1:x2]

                    if face.size > 0:

                        face = cv2.resize(face, (224,224))

                        frame_name = f"{video[:-4]}_{saved_count}.jpg"

                        save_path = os.path.join(save_dir, frame_name)

                        cv2.imwrite(save_path, face)

                        saved_count += 1

            frame_id += 1

        cap.release()

    print(class_name, "completed.")

In [ ]:
for class_name, video_folder in dataset_paths.items():

    print("\nProcessing:", class_name)

    train_videos, val_videos = split_videos(video_folder)

    # Train
    extract_faces_from_videos(
        video_folder,
        train_videos,
        os.path.join(output_root,"train"),
        class_name
    )

    # Validation
    extract_faces_from_videos(
        video_folder,
        val_videos,
        os.path.join(output_root,"val"),
        class_name
    )


Processing: original


C:\Users\jnana\anaconda3\envs\mediapipe_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 800/800 [40:53<00:00,  3.07s/it]  


original completed.


100%|██████████| 200/200 [09:08<00:00,  2.74s/it]


original completed.

Processing: deepfakes


...

100%|██████████| 200/200 [07:09<00:00,  2.15s/it]


neuraltextures completed.

Processing: faceshifter


100%|██████████| 800/800 [38:12<00:00,  2.87s/it]  


faceshifter completed.


 11%|█         | 22/200 [00:51<07:14,  2.44s/it]

In [9]:
remaining_classes = ["faceshifter", "deepfakedetection"]

for class_name in remaining_classes:

    video_folder = dataset_paths[class_name]

    print("\nProcessing:", class_name)

    train_videos, val_videos = split_videos(video_folder)

    # Train
    extract_faces_from_videos(
        video_folder,
        train_videos,
        os.path.join(output_root, "train"),
        class_name
    )

    # Validation
    extract_faces_from_videos(
        video_folder,
        val_videos,
        os.path.join(output_root, "val"),
        class_name
    )


Processing: faceshifter


C:\Users\jnana\anaconda3\envs\mediapipe_env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 800/800 [40:30<00:00,  3.04s/it] 


faceshifter completed.


100%|██████████| 200/200 [09:49<00:00,  2.95s/it]


faceshifter completed.

Processing: deepfakedetection


100%|██████████| 800/800 [46:57<00:00,  3.52s/it] 


deepfakedetection completed.


100%|██████████| 200/200 [11:15<00:00,  3.38s/it]

deepfakedetection completed.


In [ ]:
for split in ["train","val"]:
    
    print("\n", split.upper())

    for c in classes:

        path = os.path.join(output_root, split, c)

        count = len(os.listdir(path))

        print(c, count)